In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.utils.random import sample_without_replacement
import tensorflow as tf

import sklearn.decomposition as dp
import sklearn.linear_model as lm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [ ]:
sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data


In [ ]:
fnm='/media/austin/ThickBoy__1/DataAgression_Granger2/Aggression_sub_12.mat'
power,coherence,granger,labels = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])

myLabel = labels['windows']
mouse = np.asarray(myLabel['mouse'])
group = np.asarray(myLabel['group'])
expDate = np.asarray(myLabel['expDate'])
behavior = np.asarray(myLabel['behavior'])
behaviornon1 = np.asarray(myLabel['behaviornon1'])
time = np.asarray(myLabel['time'])
condition = np.asarray(myLabel['condition'])
N = len(mouse)


In [ ]:
granger = np.exp(granger)
#granger[granger>10] = 10
#power = power*10
#power[power>6] = 6

Xo = np.hstack((power,coherence,granger))
ss = StandardScaler()
X = ss.fit_transform(Xo)
#X = X[indx_tot]
#X = X - np.mean(X,axis=0)

In [ ]:
X_train,X_test = train_test_split(X,test_size=0.2,random_state=42)

In [ ]:
recon_losses_train = np.zeros(50)
recon_losses_test = np.zeros(50)

for i in range(50):
    print(i)
    model_nmf = dp.PCA(int(i+1))
    S_train = model_nmf.fit_transform(X_train)
    S_test = model_nmf.transform(X_test)
    X_r_train = np.dot(S_train,model_nmf.components_)
    X_r_test = np.dot(S_test,model_nmf.components_)
    recon_losses_train[i] = np.mean((X_train-X_r_train)**2)
    recon_losses_test[i] = np.mean((X_test-X_r_test)**2)
    

In [ ]:
plt.plot(recon_losses_test)

In [ ]:
np.savetxt('PCA_recon_train.txt',recon_losses_train,fmt='%0.8f',delimiter='\n')
np.savetxt('PCA_recon_test.txt',recon_losses_test,fmt='%0.8f',delimiter='\n')

# Load the data

In [ ]:
recon_losses_train = np.genfromtxt('PCA_recon_train.txt')
recon_losses_test = np.genfromtxt('PCA_recon_test.txt')


In [ ]:
def BIC_orig(N,p,i,loss):
    part1 = i*p*np.log(N)
    part2 = N*p*np.log(loss)
    return part1-2*part2

In [ ]:
N =  11123
p = 9856

In [ ]:
bics = np.zeros(len(recon_losses_train))
for i in range(len(recon_losses_train)):
    bics[i] = BIC_orig(N,p,i+1,recon_losses_train[i])

In [ ]:
plt.plot(bics)
plt.title('Previous BIC')

In [ ]:
def BIC_new(N,p,i,loss):
    part1 = i*p*np.log(N)
    part2 = N*p*loss
    return part1-2*part2

In [ ]:
bics = np.zeros(len(recon_losses_train))
for i in range(len(recon_losses_train)):
    bics[i] = BIC_new(N,p,i+1,recon_losses_train[i])

In [ ]:
plt.plot(bics)
plt.title('Correct BIC')